<a href="https://colab.research.google.com/github/Hassangenius/Hassan-Al-Hashimi/blob/main/Copy_of_face_emotions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.layers import Dense, Input, Activation, Flatten
from keras.utils import to_categorical

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [7]:
Data = r"C:\Users\user\downloads\archive"   # change this to the path of your extracted folder

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

for root, dirs, files in os.walk("/content"):
    if "train" in dirs and "test" in dirs:
        print("Dataset found:", root)

Dataset found: /content/archive


In [10]:
import os

print("Files in /content:")
print(os.listdir("/content"))

Files in /content:
['.config', 'drive', 'archive.zip', 'sample_data']


In [11]:
import os
import zipfile

zip_path = "/content/archive.zip"
extract_path = "/content/archive"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")
print("Extracted contents:", os.listdir(extract_path))

Extraction completed!
Extracted contents: ['train', 'test']


In [12]:
import os

Data = "/content/archive"

for root, dirs, files in os.walk(Data):
    if "train" in dirs:
        print("Train folder:", os.path.join(root, "train"))
    if "test" in dirs:
        print("Test folder:", os.path.join(root, "test"))

Train folder: /content/archive/train
Test folder: /content/archive/test


In [13]:
import os

Data = "/content/archive"

# Locate the training folder, including nested folders
train_dir = None
test_dir = None

for root, dirs, files in os.walk(Data):
    if "train" in dirs and train_dir is None:
        train_dir = os.path.join(root, "train")
    if "test" in dirs and test_dir is None:
        test_dir = os.path.join(root, "test")

if train_dir is None:
    raise FileNotFoundError(
        "Training folder not found. Please check the extracted contents."
    )

classes = sorted([
    name for name in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, name))
])

print("Training folder:", train_dir)
print("Testing folder:", test_dir)
print("Emotion classes:", classes)

Training folder: /content/archive/train
Testing folder: /content/archive/test
Emotion classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [14]:
import os

Data = "/content/archive"
train_dir = os.path.join(Data, "train")
test_dir = os.path.join(Data, "test")

classes = sorted([
    name for name in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, name))
])

print("Emotion classes:", classes)

for emotion in classes:
    train_count = len(os.listdir(os.path.join(train_dir, emotion)))
    test_count = len(os.listdir(os.path.join(test_dir, emotion)))

    print(f"{emotion}: Train = {train_count}, Test = {test_count}")

Emotion classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
angry: Train = 3995, Test = 958
disgust: Train = 436, Test = 111
fear: Train = 4097, Test = 1024
happy: Train = 7215, Test = 1774
neutral: Train = 4965, Test = 1233
sad: Train = 4830, Test = 1247
surprise: Train = 3171, Test = 831


In [15]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
import json

save_dir = "/content/drive/MyDrive/emotion_project"
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, "emotion_cnn.keras"))

with open(os.path.join(save_dir, "emotion_classes.json"), "w") as f:
    json.dump(classes, f)

print("Model and emotion classes saved to Google Drive!")

NameError: name 'model' is not defined

In [5]:
import tensorflow as tf
import os
import json

Data = "/content/archive"
train_dir = os.path.join(Data, "train")
test_dir = os.path.join(Data, "test")

IMG_SIZE = (48, 48)
BATCH_SIZE = 64
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="grayscale",
    label_mode="int",
    shuffle=True,
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="grayscale",
    label_mode="int",
    shuffle=False
)

classes = train_ds.class_names

print("Emotion classes:", classes)

Found 28709 files belonging to 7 classes.
Found 7178 files belonging to 7 classes.
Emotion classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [6]:
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(48, 48, 1)),

    layers.Rescaling(1.0 / 255),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.30),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.40),
    layers.Dense(len(classes), activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 684,423 (2.61 MB)

 Trainable params: 683,975 (2.61 MB)

 Non-trainable params: 448 (1.75 KB)

In [2]:
import tensorflow as tf
import os
import json

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


In [7]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "/content/drive/MyDrive/emotion_cnn_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True
    )
]

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=3,
    callbacks=callbacks
)

Epoch 1/3
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 510ms/step - accuracy: 0.2304 - loss: 2.0788
Epoch 1: val_accuracy improved from None to 0.26233, saving model to /content/drive/MyDrive/emotion_cnn_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/emotion_cnn_best.keras
449/449 ━━━━━━━━━━━━━━━━━━━━ 245s 537ms/step - accuracy: 0.2546 - loss: 1.8500 - val_accuracy: 0.2623 - val_loss: 1.8112
Epoch 2/3
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 511ms/step - accuracy: 0.3295 - loss: 1.6639
Epoch 2: val_accuracy improved from 0.26233 to 0.42588, saving model to /content/drive/MyDrive/emotion_cnn_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/emotion_cnn_best.keras
449/449 ━━━━━━━━━━━━━━━━━━━━ 242s 538ms/step - accuracy: 0.3430 - loss: 1.6391 - val_accuracy: 0.4259 - val_loss: 1.5075
Epoch 3/3
449/449 ━━━━━━━━━━━━━━━━━━━━ 0s 497ms/step - accuracy: 0.3785 - loss: 1.5640
Epoch 3: val_accuracy improved from 0.42588 to 0.42742, saving model to /content/drive/MyDrive/emotion

In [8]:
save_dir = "/content/drive/MyDrive/emotion_project"
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, "emotion_cnn.keras"))

with open(
    os.path.join(save_dir, "emotion_classes.json"), "w"
) as f:
    json.dump(classes, f)

print("Model and class labels saved successfully!")
print("Location:", save_dir)

Model and class labels saved successfully!
Location: /content/drive/MyDrive/emotion_project


In [9]:
import os
import json
import tensorflow as tf

save_dir = "/content/drive/MyDrive/emotion_project"

model_path = os.path.join(save_dir, "emotion_cnn.keras")
labels_path = os.path.join(save_dir, "emotion_classes.json")

print("Model exists:", os.path.exists(model_path))
print("Labels exist:", os.path.exists(labels_path))

model = tf.keras.models.load_model(model_path)

with open(labels_path, "r") as f:
    classes = json.load(f)

print("Model loaded successfully!")
print("Emotion classes:", classes)

Model exists: True
Labels exist: True
Model loaded successfully!
Emotion classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [10]:

import gradio as gr
import numpy as np
from PIL import Image

def predict_emotion(image):
    if image is None:
        return {}

    # Preprocess the image to match the CNN input
    image = image.convert("L").resize((48, 48))
    image_array = np.array(image, dtype=np.float32)
    image_array = np.expand_dims(image_array, axis=(0, -1))

    # Predict probabilities
    predictions = model.predict(image_array, verbose=0)[0]

    # Return emotion probabilities for Gradio
    return {
        classes[i]: float(predictions[i])
        for i in range(len(classes))
    }

demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Image(type="pil", label="Upload a face image"),
    outputs=gr.Label(num_top_classes=7, label="Emotion predictions"),
    title="Facial Emotion Recognition",
    description=(
        "Upload a face image to predict one of seven emotion categories."
    )
)

demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2b7cca5ac386404712.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
